# Notebook 04 — Baseline Model Training

Trains and compares **ResNet50** and **ViT-Base** baselines for the Hebrew PDF hallucination-risk classifier.

Both models are trained with `BCEWithLogitsLoss` + `AdamW`, then post-hoc temperature-calibrated and evaluated with full metrics.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.data.dataset import HallucinationRiskDataset
from src.models.resnet_baseline import ResNetClassifier
from src.models.vit_baseline import ViTClassifier
from src.models.calibrator import TemperatureCalibrator
from src.utils.metrics import compute_metrics, compute_per_institution_metrics, compute_ece
from src.utils.visualization import (
    plot_confusion_matrix, plot_roc_curve, plot_pr_curve,
    plot_score_distribution
)

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT          = Path('..').resolve()
METADATA_CSV  = ROOT / 'data' / 'metadata.csv'
RENDERED_DIR  = ROOT / 'data' / 'rendered'
SPLITS_DIR    = ROOT / 'data' / 'splits'
CHECKPOINT_DIR = ROOT / 'checkpoints' / 'baseline'
LOG_DIR        = ROOT / 'logs' / 'baseline'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print(f'Device: {device}')

## 1 — Load Config


In [ ]:
with open(ROOT / 'configs' / 'baseline.yaml') as f:
    cfg = yaml.safe_load(f)

print(yaml.dump(cfg, default_flow_style=False))

# ── Pull out key hypers ─────────────────────────────────────────────────────
BATCH_SIZE = cfg['training']['batch_size']
LR         = cfg['training']['learning_rate']
WD         = cfg['training']['weight_decay']
PATIENCE   = cfg['training']['early_stopping_patience']
EPOCHS     = 5   # quick baseline; set to cfg['training']['epochs'] for full run

## 2 — Dataset Inspection


In [ ]:
train_ds = HallucinationRiskDataset(METADATA_CSV, 'train', RENDERED_DIR, augment=True)
val_ds   = HallucinationRiskDataset(METADATA_CSV, 'val',   RENDERED_DIR, augment=False)
test_ds  = HallucinationRiskDataset(METADATA_CSV, 'test',  RENDERED_DIR, augment=False)

print(f'Train size : {len(train_ds)}')
print(f'Val size   : {len(val_ds)}')
print(f'Test size  : {len(test_ds)}')

# ── Label balance ────────────────────────────────────────────────────────────
train_labels = np.array([train_ds[i][1].item() for i in range(len(train_ds))])
val_labels   = np.array([val_ds[i][1].item()   for i in range(len(val_ds))])
print(f'\nTrain label distribution  — safe: {(train_labels==0).sum()}, risky: {(train_labels==1).sum()}')
print(f'Val   label distribution  — safe: {(val_labels==0).sum()}, risky: {(val_labels==1).sum()}')

## 3 — DataLoaders


In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=(device.type=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)}')
print(f'Val   batches: {len(val_loader)}')

# Sanity-check a single batch shape
images, labels = next(iter(train_loader))
print(f'Batch images shape: {images.shape}  — dtype: {images.dtype}')
print(f'Batch labels shape: {labels.shape}')

## 4 — Training Utility

Shared training loop for both baselines.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, n = 0.0, 0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        optimizer.zero_grad()
        logits = model(images).squeeze(1)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(labels)
        n += len(labels)
    return total_loss / n


@torch.no_grad()
def evaluate_epoch(model, loader, criterion, device):
    """Returns (val_loss, all_logits_np, all_labels_np)."""
    model.eval()
    total_loss, n = 0.0, 0
    all_logits, all_labels = [], []
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.float().to(device, non_blocking=True)
        logits = model(images).squeeze(1)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        n += len(labels)
        all_logits.append(logits.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    logits_np = np.concatenate(all_logits)
    labels_np = np.concatenate(all_labels)
    return total_loss / n, logits_np, labels_np


def run_training(model, train_loader, val_loader, device, epochs=EPOCHS, lr=LR, wd=WD, name='model'):
    """Full training loop with loss-curve plotting. Returns (model, val_logits, val_labels)."""
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)

    train_losses, val_losses = [], []
    best_val_loss = float('inf')
    best_state = None

    for epoch in range(1, epochs + 1):
        tr_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_logits, val_labels = evaluate_epoch(model, val_loader, criterion, device)
        train_losses.append(tr_loss)
        val_losses.append(val_loss)
        print(f'[{name}] Epoch {epoch}/{epochs}  train_loss={tr_loss:.4f}  val_loss={val_loss:.4f}')
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state    = {k: v.clone() for k, v in model.state_dict().items()}

    # Reload best weights
    model.load_state_dict(best_state)

    # Plot loss curves
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(range(1, epochs + 1), train_losses, label='Train loss')
    ax.plot(range(1, epochs + 1), val_losses,   label='Val loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('BCE Loss')
    ax.set_title(f'{name} — Loss Curves')
    ax.legend()
    plt.tight_layout()
    plt.show()

    # Final val logits/labels from best model
    _, val_logits, val_labels = evaluate_epoch(model, val_loader, criterion, device)
    return model, val_logits, val_labels

## 5 — ResNet50 Baseline Training


In [ ]:
resnet = ResNetClassifier(model_name='resnet50', pretrained=True, num_classes=1).to(device)
total_params    = sum(p.numel() for p in resnet.parameters())
trainable_params = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
print(f'ResNet50  total params: {total_params:,}  trainable: {trainable_params:,}')

resnet, resnet_val_logits, resnet_val_labels = run_training(
    resnet, train_loader, val_loader, device,
    epochs=EPOCHS, lr=LR, wd=WD, name='ResNet50'
)

## 6 — ResNet50 Evaluation


In [ ]:
# ── Temperature calibration ──────────────────────────────────────────────────
resnet_cal = TemperatureCalibrator()
resnet_cal.calibrate(resnet_val_logits, resnet_val_labels)
print(f'ResNet50 temperature: {resnet_cal.temperature:.4f}')

resnet_val_probs  = resnet_cal.predict(resnet_val_logits)
resnet_thresholds = resnet_cal.get_thresholds(
    resnet_val_probs, resnet_val_labels, target_false_safe_rate=0.05
)
print(f'T_low={resnet_thresholds["T_low"]:.3f}  T_high={resnet_thresholds["T_high"]:.3f}')

# ── Metrics ─────────────────────────────────────────────────────────────────
resnet_metrics = compute_metrics(resnet_val_labels, resnet_val_probs, resnet_thresholds)
print('\nResNet50 Val Metrics:')
for k, v in resnet_metrics.items():
    print(f'  {k:25s}: {v:.4f}')

# ── Score distribution ───────────────────────────────────────────────────────
plot_score_distribution(
    resnet_val_probs, resnet_val_labels.astype(int),
    output_path=str(LOG_DIR / 'resnet_score_dist.png')
)
from IPython.display import Image as IPImage
IPImage(str(LOG_DIR / 'resnet_score_dist.png'))

## 7 — ViT-Base Baseline Training


In [ ]:
vit = ViTClassifier(model_name='vit_base_patch16_224', pretrained=True, num_classes=1).to(device)
total_params_vit    = sum(p.numel() for p in vit.parameters())
trainable_params_vit = sum(p.numel() for p in vit.parameters() if p.requires_grad)
print(f'ViT-Base  total params: {total_params_vit:,}  trainable: {trainable_params_vit:,}')

vit, vit_val_logits, vit_val_labels = run_training(
    vit, train_loader, val_loader, device,
    epochs=EPOCHS, lr=LR, wd=WD, name='ViT-Base'
)

## 8 — ViT Evaluation


In [ ]:
# ── Temperature calibration ──────────────────────────────────────────────────
vit_cal = TemperatureCalibrator()
vit_cal.calibrate(vit_val_logits, vit_val_labels)
print(f'ViT-Base temperature: {vit_cal.temperature:.4f}')

vit_val_probs  = vit_cal.predict(vit_val_logits)
vit_thresholds = vit_cal.get_thresholds(
    vit_val_probs, vit_val_labels, target_false_safe_rate=0.05
)
print(f'T_low={vit_thresholds["T_low"]:.3f}  T_high={vit_thresholds["T_high"]:.3f}')

# ── Metrics ─────────────────────────────────────────────────────────────────
vit_metrics = compute_metrics(vit_val_labels, vit_val_probs, vit_thresholds)
print('\nViT-Base Val Metrics:')
for k, v in vit_metrics.items():
    print(f'  {k:25s}: {v:.4f}')

# ── Score distribution ───────────────────────────────────────────────────────
plot_score_distribution(
    vit_val_probs, vit_val_labels.astype(int),
    output_path=str(LOG_DIR / 'vit_score_dist.png')
)
IPImage(str(LOG_DIR / 'vit_score_dist.png'))

## 9 — Model Comparison


In [ ]:
comparison_rows = []
for name, metrics in [('ResNet50', resnet_metrics), ('ViT-Base', vit_metrics)]:
    comparison_rows.append({
        'model':           name,
        'f1':              metrics['f1'],
        'false_safe_rate': metrics['false_safe_rate'],
        'review_rate':     metrics['review_rate'],
        'roc_auc':         metrics['roc_auc'],
        'pr_auc':          metrics['pr_auc'],
        'ece':             metrics['ece'],
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('model')
print('\nBaseline Comparison (Val Set):')
print(comparison_df.to_string(float_format='{:.4f}'.format))

# ── Side-by-side bar chart ───────────────────────────────────────────────────
metrics_to_plot = ['f1', 'false_safe_rate', 'review_rate', 'roc_auc']
x = np.arange(len(metrics_to_plot))
width = 0.3

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, [resnet_metrics[m] for m in metrics_to_plot], width, label='ResNet50', color='steelblue')
ax.bar(x + width/2, [vit_metrics[m]    for m in metrics_to_plot], width, label='ViT-Base',  color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylabel('Score')
ax.set_title('ResNet50 vs ViT-Base — Key Metrics (Val Set)')
ax.legend()
plt.tight_layout()
plt.show()

## Conclusion

**Key observations:**
- Both models were trained for 5 epochs as a quick baseline; `cfg['training']['epochs']` (30) should be used for a full run.
- The model with lower `false_safe_rate` is preferred since this is our primary safety constraint (target ≤ 5%).
- `review_rate` should remain below ~30% to be operationally viable.
- The better baseline serves as the performance floor that DiT (`05_dit_training.ipynb`) must beat.


## 10 — Save Checkpoints


In [ ]:
# ── ResNet50 ─────────────────────────────────────────────────────────────────
resnet_ckpt = {
    'model_state_dict': resnet.state_dict(),
    'thresholds': resnet_thresholds,
    'metrics':    resnet_metrics,
    'temperature': resnet_cal.temperature,
}
torch.save(resnet_ckpt, CHECKPOINT_DIR / 'resnet50_best.pt')
resnet_cal.save(str(CHECKPOINT_DIR / 'resnet50_calibrator.pkl'))
print(f'ResNet50 checkpoint saved → {CHECKPOINT_DIR / "resnet50_best.pt"}')

# ── ViT-Base ─────────────────────────────────────────────────────────────────
vit_ckpt = {
    'model_state_dict': vit.state_dict(),
    'thresholds': vit_thresholds,
    'metrics':    vit_metrics,
    'temperature': vit_cal.temperature,
}
torch.save(vit_ckpt, CHECKPOINT_DIR / 'vit_base_best.pt')
vit_cal.save(str(CHECKPOINT_DIR / 'vit_base_calibrator.pkl'))
print(f'ViT-Base  checkpoint saved → {CHECKPOINT_DIR / "vit_base_best.pt"}')

# ── Persist metrics for notebook 05 ──────────────────────────────────────────
comparison_df.to_csv(CHECKPOINT_DIR / 'baseline_comparison.csv')
print('Comparison table saved → baseline_comparison.csv')